# 04 — Controlled SQL Analytics

In [6]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

Project root: /Users/B1/ghome/github/online/reservation-analytics-ai-agent


## Goal

Understand why the application resolves Campaign + Product + Country before executing controlled analytics SQL.

**Q&A — Why did you use controlled SQL instead of letting the LLM generate SQL?**

Answer:

I intentionally separated natural-language understanding from SQL execution. The LLM only extracts structured business context, such as campaign, product, country, and metric. The application then validates that context and maps it to predefined SQL logic. This reduces hallucinated SQL, incorrect joins, and security risks.

One improvement I would make for production is to replace manual value **quoting with parameterized queries** where the backend supports them. That would make the SQL safer and easier to maintain while keeping the query structure controlled by the application.

In [3]:
from app.settings import load_settings
from app.data.backend import create_backend
from app.analytics.resolver import CampaignResolver
from app.analytics.service import AnalyticsService
from app.core.models import ReservationQuery

settings = load_settings('config/local.env')
backend = create_backend(settings)
resolver = CampaignResolver(backend)
analytics = AnalyticsService(backend)

In [4]:
query = ReservationQuery(
    country='Germany',
    product='Phone Mi 17 Pro',
    campaign_id='CMP001',
)
campaigns = resolver.resolve(query)
print(json.dumps([item.model_dump() for item in campaigns], indent=2, ensure_ascii=False))


[
  {
    "campaign_id": "CMP001",
    "campaign_name": "Phone Mi 17 Pro Launch",
    "product_id": "P001",
    "product_name": "Phone Mi 17 Pro",
    "country_code": "DE",
    "country_name": "Germany",
    "start_time": "2026-08-01 00:00:00",
    "end_time": "2026-08-15 23:59:59"
  }
]


In [5]:
campaign = campaigns[0]
print(analytics.run('reserved_users', campaign))
print(analytics.run('conversion_rate', campaign))

CMP001 — Phone Mi 17 Pro Launch (Germany): 8 reserved users.
CMP001 — Phone Mi 17 Pro Launch (Germany): reservation-to-order conversion rate was 62.50%.


Key design rule:

- LLM: understand user intent and context.
- Application: select controlled SQL.
- Data Mart: provide trusted metrics or detail records.